# ESM-2 model usage with `transformers`

notebook based on: https://github.com/huggingface/notebooks/blob/main/examples/protein_language_modeling.ipynb
new elements:
- sequence classification by PEFT
- sequence classification with KNN using ESM-2 as feature extractor
- sequence classification with custom model

**Rodzina modeli ESM-2**

ESM-2 występuje w kilku rozmiarach modelu bazowego, od 8 mln do 8 mld parametrów. Tutaj będziemy wykorzystywać mały model 35M https://huggingface.co/facebook/esm2_t12_35M_UR50D

Wersja modelu bazowego jest określona w zmiennej `model_checkpoint = "facebook/esm2_t12_35M_UR50D"`

Hugging Face oferuje różne architektury zadaniowe (task-specific architectures) dla danego modelu bazowego, z których każda wykorzystuje inną głowę/głowicę modelu (head) do konkretnych zadań końcowych

## 1. Set-up, check if works
install, download, initialize tokenizer and model, check what is the output for a given sequence

In [ ]:
!uv pip install evaluate peft==0.18

In [ ]:
from transformers import AutoTokenizer, EsmModel
import torch

model_checkpoint = "facebook/esm2_t12_35M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = EsmModel.from_pretrained(model_checkpoint)

sequence = "MAKWGEGDPRWIVEERADATNVNNWHWTERDASNWSTDKLKTLFLAVQVQNEEGKCEVTEVSKLDGEASINNRKGKLIFFYEWSV"
inputs = tokenizer(sequence, return_tensors="pt")
outputs = model(**inputs)
print("Hidden state shape:", outputs.last_hidden_state.shape, "(batch_size, sequence_length, hidden_size)")
print("Pooled output shape:", outputs.pooler_output.shape, "(batch_size, hidden_size)")

In [ ]:
# Co zwraca tokenizer?
print(inputs)

for key, value in inputs.items():
    print(f"{key}: {value.shape} (batch size: {value.shape[0]}, sequence length: {value.shape[1]})")

`input_ids` to indeksy tokenów, `attention_mask` zawiera 1, jeśli dany token jest ważny i 0, jeśli należy go pominąć (potrzebne np w paddingu)

In [ ]:
print("Długość sekwencji:", len(sequence))
print("Długość tokenów:", len(tokenizer.encode(sequence)))
print(tokenizer.decode(inputs["input_ids"][0]))
print("Wielkość słownika:", tokenizer.vocab_size)

In [ ]:
# Get the full vocabulary dictionary
vocab = tokenizer.get_vocab()
for key, value in vocab.items():
    print(f"{key}: {value}")

- `.` and `-` are used in protein sequencies
- `<eos>` is the special token used to indicate the end of a sequence
- `<unk>` is the special token used for unknown or out-of-vocabulary tokens that are not present in the tokenizer's vocabulary
- `<mask>` is the special token used for masked language modeling tasks, where certain tokens in the input sequence are replaced with <mask> and the model is trained to predict the original token
- `<null_1>` is an unused special token that may be reserved for future use or specific tasks, but it does not have a predefined meaning in the context of the ESM model

## 2. Run as is
mask selected amino-acids and check the output probabilities

In [ ]:
# token maski
print("Token maski i jego ID:", tokenizer.mask_token, tokenizer.mask_token_id)

print("Dekodowany token maski:", tokenizer.decode(tokenizer.mask_token_id))

print("Kodowany token maski:", tokenizer.encode("<mask>"), "(początek sekwencji, token maski, koniec sekwencji)")

In [ ]:
from transformers import EsmForMaskedLM

masked_lm = EsmForMaskedLM.from_pretrained(model_checkpoint)
sequence_with_mask = "MAKWGEGDPRWIVEERADATNVNNWHWTERDASNWSTDKLKTLFLAVQVQNEEGKCEVTEVSKLDGEASINNRKGKLIFFYEWS<mask>"
inputs = tokenizer(sequence_with_mask, return_tensors="pt")


with torch.no_grad():
    outputs = masked_lm(**inputs)
logits = outputs.logits

print("Logits shape:", logits.shape, "(batch size, sequence length, vocab size)")

mask_token_indices = (inputs.input_ids == tokenizer.mask_token_id)[0].nonzero(as_tuple=True)[0]
predicted_tokens_id = logits[0, mask_token_indices].argmax(axis=-1)
for token_index, predicted_token_id in zip(mask_token_indices, predicted_tokens_id):
    print(f"Predicted token ID for position {token_index.item()-1}: {predicted_token_id.item()}")


## 3. Fine-tune for sequence classification
as in `protein_language_modeling.ipynb`

In [ ]:
import requests
from io import BytesIO
import pandas as pd

query_url ="https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Csequence%2Ccc_subcellular_location&format=tsv&query=%28%28organism_id%3A9606%29%20AND%20%28reviewed%3Atrue%29%20AND%20%28length%3A%5B80%20TO%20500%5D%29%29"
uniprot_request = requests.get(query_url)
bio = BytesIO(uniprot_request.content)
df = pd.read_csv(bio, compression='gzip', sep='\t')

df = df.dropna()
cytosolic = df['Subcellular location [CC]'].str.contains("Cytosol") | df['Subcellular location [CC]'].str.contains("Cytoplasm")
membrane = df['Subcellular location [CC]'].str.contains("Membrane") | df['Subcellular location [CC]'].str.contains("Cell membrane")

cytosolic_df = df[cytosolic & ~membrane]
membrane_df = df[membrane & ~cytosolic]

cytosolic_sequences = cytosolic_df["Sequence"].tolist()
cytosolic_labels = [0 for _ in cytosolic_sequences]
membrane_sequences = membrane_df["Sequence"].tolist()
membrane_labels = [1 for _ in membrane_sequences]

sequences = cytosolic_sequences + membrane_sequences
labels = cytosolic_labels + membrane_labels

print("Liczba sekwencji cytosolowych:", len(cytosolic_sequences))
print("Liczba sekwencji membranowych:", len(membrane_sequences))

In [ ]:
print("przykładowa sekwencja:", sequences[0])
print("przykładowa etykieta:", labels[0])

In [ ]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

train_sequences, test_sequences, train_labels, test_labels = train_test_split(sequences, labels, test_size=0.25, shuffle=True)
train_tokenized = tokenizer(train_sequences)
test_tokenized = tokenizer(test_sequences)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict(train_tokenized).add_column("labels", train_labels)
test_dataset = Dataset.from_dict(test_tokenized).add_column("labels", test_labels)

In [ ]:
print("Dataset zwraca słownik z kluczami:", train_dataset.column_names)
print("Przykładowy element datasetu:", train_dataset[0])


Potrzebujemy zrobić konwersję modelu z przeiwdywania zasłoniętego tokenu na klasyfikację. Z `transformers` można to zrobić jednym poleceniem `AutoModelForSequenceClassification.from_pretrained`

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

num_labels = 2
clf_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

args = TrainingArguments(
    "esm2-finetuned-localization",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = Trainer(
    clf_model,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)



In [ ]:
trainer.train()

### 3.1 Inference using the fine-tuned model
Let's load the model from the checkpoint and predict the localization of a sample sequence.

In [ ]:
from transformers import pipeline
import torch.nn.functional as F

# Load the model from the local save directory
clf_pipeline = pipeline("sentiment-analysis", model="esm2-finetuned-localization/checkpoint-488", tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

sample_seq = "MAKWGEGDPRWIVEERADATNVNNWHWTERDASNWSTDKLKTLFLAVQVQNEEGKCEVTEVSKLDGEASINNRKGKLIFFYEWSV"
result = clf_pipeline(sample_seq)[0]

label = 'Membrane' if int(result['label'].split('_')[-1]) == 1 else 'Cytosolic'
print(f"Sequence: {sample_seq[:20]}...")
print(f"Predicted Class: {label}")
print(f"Confidence Score: {result['score']:.4f}")

## 4. PEFT for sequence classification
as before, but with PEFT

In [ ]:
print(clf_model)

Jak wybrać warstwy do modyfikacji?

Zwykle wszystkie `query` i `value`, czasami również `key` i `dense`

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, # Sequence Classification
    inference_mode=False, # False for training, True for inference
    r=8, # rank
    lora_alpha=16, # Scaling Factor. Low rank adapter is multiplied by alpha/r and added to weights
    lora_dropout=0.1,
    target_modules=["query", "value"] # ESM attention modules
)

peft_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)
peft_model = get_peft_model(peft_model, peft_config)
peft_model.print_trainable_parameters()

In [ ]:
peft_args = TrainingArguments(
    "esm2-peft-localization",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=1e-3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

peft_trainer = Trainer(
    peft_model,
    peft_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

peft_trainer.train()

### 4.1 Inference using the PEFT model
We load the PEFT adapter and run a prediction.

In [ ]:
from peft import PeftModel, PeftConfig
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load base model and the trained adapter
peft_model_path = "esm2-peft-localization/checkpoint-488"
config = PeftConfig.from_pretrained(peft_model_path)
base_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)
loaded_peft_model = PeftModel.from_pretrained(base_model, peft_model_path)
loaded_peft_model.to(device).eval()

inputs = tokenizer(sample_seq, return_tensors="pt").to(device)
with torch.no_grad():
    logits = loaded_peft_model(**inputs).logits
    probabilities = F.softmax(logits, dim=-1)
    prediction = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][prediction].item()

label = 'Membrane' if prediction == 1 else 'Cytosolic'
print(f"PEFT Predicted Class: {label}")
print(f"Confidence Score: {confidence:.4f}")

## 5. Fine tune for token classification


In [ ]:
# Secondary structure prediction (Token classification)
query_url ="https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession%2Csequence%2Cft_helix%2Cft_strand%2Cft_turn&format=tsv&query=%28%28organism_id%3A9606%29%20AND%20%28reviewed%3Atrue%29%20AND%20%28length%3A%5B80%20TO%20500%5D%29%29"
uniprot_request = requests.get(query_url)
bio = BytesIO(uniprot_request.content)
df = pd.read_csv(bio, compression='gzip', sep='\t')

In [ ]:
no_structure_rows = df["Beta strand"].isna() & df["Helix"].isna()
df = df[~no_structure_rows]
df

In [ ]:
import re
import numpy as np

def get_positions(x):
    positions = []
    if pd.isna(x):
        return positions
    # Modified regex: ensure capturing groups for numbers are always present
    for match in re.finditer(r"(?:STRAND|HELIX|TURN)\s+(\d+)\.\.(\d+)", x):
        try:
            start, end = int(match.group(1)), int(match.group(2))
            positions.append((start, end))
        except ValueError:
            # This except block will now only catch cases where the captured groups
            # are not valid integers, which is less likely with the corrected regex.
            pass
    return positions

df["helix_pos"] = df["Helix"].apply(get_positions)
df["strand_pos"] = df["Beta strand"].apply(get_positions)
df["turn_pos"] = df["Turn"].apply(get_positions)

def generate_labels(row):
    seq_len = len(row["Sequence"])
    labels = np.zeros(seq_len, dtype=int)

    # 0 - uknown, 1 - helix, 2 - strand, 3- turn
    for start, end in row["helix_pos"]:
        if start - 1 < seq_len and end <= seq_len:
            labels[start-1:end] = 1
    for start, end in row["strand_pos"]:
        if start - 1 < seq_len and end <= seq_len:
            labels[start-1:end] = 2
    for start, end in row["turn_pos"]:
        if start - 1 < seq_len and end <= seq_len:
            labels[start-1:end] = 3
    return labels.tolist()

df["labels"] = df.apply(generate_labels, axis=1)

token_sequences = df["Sequence"].tolist()
token_labels = df["labels"].tolist()

In [ ]:
print(f"Przykładowa sekwencja ({len(token_sequences[0])}):", token_sequences[0])
print(f"Przykładowe etykiety ({len(token_labels[0])}):", token_labels[0])

In [ ]:
train_token_seqs, test_token_seqs, train_token_labs, test_token_labs = train_test_split(
    token_sequences, token_labels, test_size=0.25, shuffle=True
)

train_token_tokenized = tokenizer(train_token_seqs)
test_token_tokenized = tokenizer(test_token_seqs)

# Pad labels to match sequence lengths (accounting for special tokens [CLS] and [EOS])
def align_labels(tokenized, labels):
    aligned_labels = []
    for i in range(len(labels)):
        seq_labels = labels[i]
        input_ids = tokenized["input_ids"][i]

        # -100 is the ignore index for PyTorch CrossEntropyLoss
        aligned = [-100] + seq_labels + [-100]
        aligned_labels.append(aligned)
    return aligned_labels

train_token_tokenized["labels"] = align_labels(train_token_tokenized, train_token_labs)
test_token_tokenized["labels"] = align_labels(test_token_tokenized, test_token_labs)

train_token_dataset = Dataset.from_dict(train_token_tokenized)
test_token_dataset = Dataset.from_dict(test_token_tokenized)

In [ ]:
from transformers import AutoModelForTokenClassification
from transformers import DataCollatorForTokenClassification


# 0: Coil, 1: Helix, 2: Strand, 3: Turn
token_model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=4)

token_args = TrainingArguments(
    "esm2-finetuned-secondary-structure",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",

)

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy_metric(eval_pred):
    predictions, labels = eval_pred
    labels = labels.reshape((-1,))
    predictions = np.argmax(predictions, axis=2)
    predictions = predictions.reshape((-1,))
    predictions = predictions[labels!=-100] # -100 is the ignore index for PyTorch CrossEntropyLoss
    labels = labels[labels!=-100] # as above
    return accuracy_metric.compute(predictions=predictions, references=labels)

data_collator = DataCollatorForTokenClassification(tokenizer)


token_trainer = Trainer(
    token_model,
    token_args,
    train_dataset=train_token_dataset,
    eval_dataset=test_token_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_accuracy_metric
,
)

token_trainer.train()

### 5.1 Inference for Token Classification
Let's visualize the secondary structure prediction for a sequence.

In [ ]:
token_clf_pipeline = pipeline("token-classification", model="esm2-finetuned-secondary-structure/checkpoint-412", tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# 0: Coil, 1: Helix, 2: Strand, 3: Turn
label_map = {0: "C", 1: "H", 2: "S", 3: "T"}
sample_token_seq = "MEGLRRGLSRWKRYHIKVHLADEALLLPLTVRPRDTLSDLRAQLVGQGVSSWKRAFYYNARRLDDHQTVRDARLQDGSVLLLVSDPR"
predictions = token_clf_pipeline(sample_token_seq)

# Reconstruct structure string and collect avg confidence
structure = [" "] * len(sample_token_seq)
scores = []
for pred in predictions:
    idx = pred['index'] - 1
    if 0 <= idx < len(structure):
        label_id = int(pred['entity'].split('_')[-1])
        structure[idx] = label_map.get(label_id, "?")
        scores.append(pred['score'])

print(f"Seq: {sample_token_seq}")
print(f"Str: {''.join(structure)}")
print(f"Average Token Confidence: {np.mean(scores):.4f}")
print("confidence by token:", [round(x.item(), 2) for x in scores])

## 6. Protein localization classification using KNN on ESM-2 embeddings
Using the pre-trained ESM-2 model to extract sequence embeddings and training a K-Nearest Neighbors classifier.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import torch
from tqdm.auto import tqdm
import numpy as np

# We will use the base ESM-2 model loaded in section 1 and the localization dataset from section 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

def get_sequence_embeddings(seqs, batch_size=8):
    embeddings = []
    for i in tqdm(range(0, len(seqs), batch_size)):
        batch_seqs = seqs[i:i+batch_size]
        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            # Use the mean of the hidden states (excluding padding tokens)
            attention_mask = inputs['attention_mask'].unsqueeze(-1)
            sum_embeddings = torch.sum(outputs.last_hidden_state * attention_mask, dim=1)
            sum_mask = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
            mean_embeddings = sum_embeddings / sum_mask
            embeddings.append(mean_embeddings.cpu().numpy())
    return np.concatenate(embeddings, axis=0)

# Extract embeddings
print("Extracting training embeddings...")
X_train = get_sequence_embeddings(train_sequences)
print("Extracting testing embeddings...")
X_test = get_sequence_embeddings(test_sequences)

# Train KNN
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(X_train, train_labels)

# Predict and evaluate
y_pred = knn.predict(X_test)
accuracy = accuracy_score(test_labels, y_pred)
print(f"KNN Accuracy on ESM-2 embeddings: {accuracy:.4f}")

In [ ]:
# Train KNN
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(X_train, train_labels)

# Predict and evaluate
y_pred = knn.predict(X_test)
accuracy = accuracy_score(test_labels, y_pred)
print(f"KNN Accuracy on ESM-2 embeddings: {accuracy:.4f}")

## 7. Custom classifier

Ponownie klasyfikacja lokalizacji białka, tym razem z użyciem własnego modelu

In [ ]:
from torch import nn

num_labels = 2 # 0: Cytosolic, 1: Membrane

class SimpleEsmClassifier(nn.Module):
    def __init__(self, backbone, num_labels, dropout=0.2, freeze_backbone=False):
        super().__init__()
        self.num_labels = num_labels
        self.backbone = backbone
        hidden_size = self.backbone.config.hidden_size

        # Optionally freeze the backbone parameters, reducing memory usage and speeding up training
        # However, it would be better to have larger self.classifier capacity in this case
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels) # you may change it to multi layer network e.g. nn.Sequential(nn.Linear(hidden_size, hidden_size//2), nn.GELU(), nn.Linear(hidden_size//2, num_labels))
        self.loss_fn = nn.CrossEntropyLoss() # standard loss for multi-class classification

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # Sequence-level representation
        pooled = outputs.pooler_output # shape: (batch_size, hidden_size)
        logits = self.classifier(self.dropout(pooled)) # shape: (batch_size, num_labels)

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

backbone = EsmModel.from_pretrained(model_checkpoint)

custom_classifier = SimpleEsmClassifier(backbone, num_labels, freeze_backbone=False)

In [ ]:
args = TrainingArguments(
    "esm2-finetuned-localization-custom",
    eval_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = Trainer(
    custom_classifier,
    args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

### 7.1 Inference with Custom Classifier
Demonstrating how to load the state dictionary for the custom class.

In [ ]:
# To load a custom model, we re-initialize it and load the weights
import torch.nn.functional as F

custom_classifier.eval()
inputs = tokenizer(sample_seq, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = custom_classifier(**inputs)
    logits = outputs['logits']
    probabilities = F.softmax(logits, dim=-1)
    prediction = torch.argmax(probabilities, dim=-1).item()
    confidence = probabilities[0][prediction].item()

label = 'Membrane' if prediction == 1 else 'Cytosolic'
print(f"Custom Model Prediction: {label}")
print(f"Confidence Score: {confidence:.4f}")